# Spark Project: E-commerce Data Pipeline at Scale


## Problem statement

You are a data engineer on an e-commerce platform.

**Deliver:**
- Process **large** order-like data
- Clean / transform
- Produce **analytics**

**Goal:** a **scalable** Spark batch pipeline (local demo).


## Prerequisites

- PySpark + JDK ([`README.md`](README.md)).
- This notebook generates **~1M rows** — if memory is tight, lower `range(...)`.
- Writes **`output_parquet/`** next to this notebook (gitignored).


## Dataset setup (simulated scale)

We don’t ship multi-GB files in-repo — **`spark.range`** simulates volume.


In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, expr, rand

spark = (
    SparkSession.builder.appName("EcommerceSparkProject")
    .master("local[*]")
    .getOrCreate()
)

# Simulate orders — tune N if your laptop struggles
N = 1_000_000

df = spark.range(N)

df = (
    df.withColumn("user_id", (rand() * 1000).cast("int"))
    .withColumn("amount", (rand() * 1000).cast("int"))
    .withColumn(
        "category",
        expr("CASE WHEN amount > 500 THEN 'Electronics' ELSE 'Fashion' END"),
    )
)

df.show(5, truncate=False)


## Transformation pipeline


In [ ]:
from pyspark.sql.functions import upper

cleaned = df.select(
    col("id"),
    col("user_id"),
    col("amount"),
    upper(col("category")).alias("category"),
)


## Optimization (before heavy passes)

**Cache** when you’ll reuse the DataFrame; **repartition** when downstream stages benefit from target partition count (trade shuffle cost vs parallelism).


In [ ]:
cleaned = cleaned.repartition(10)
cleaned.cache()

# Materialize cache once (pick an action)
print("rows:", cleaned.count())


## Aggregations


In [ ]:
user_revenue = cleaned.groupBy("user_id").sum("amount")

category_revenue = cleaned.groupBy("category").sum("amount")

user_revenue.show(10, truncate=False)
category_revenue.show(truncate=False)


## Window function (top spenders)


In [ ]:
from pyspark.sql.functions import rank
from pyspark.sql.window import Window

window = Window.orderBy(col("sum(amount)").desc())

ranked_users = user_revenue.withColumn("rank", rank().over(window))

ranked_users.show(10, truncate=False)


## Save output (Parquet)


In [ ]:
import shutil

out_dir = "output_parquet"
shutil.rmtree(out_dir, ignore_errors=True)

cleaned.write.mode("overwrite").parquet(out_dir)

print(f"Wrote Parquet dataset to {out_dir}/")


## Analytics (Spark SQL)


In [ ]:
cleaned.createOrReplaceTempView("orders")

spark.sql(
    """
SELECT category, SUM(amount) as revenue
FROM orders
GROUP BY category
"""
).show(truncate=False)


## Practice

1. **Top 10 users** by revenue (`ranked_users` or `ORDER BY sum(amount) DESC LIMIT 10`).
2. **Average order value** — `avg(amount)` globally or per user.
3. **Count** orders per category (`COUNT(*)` with `GROUP BY category`).


## Assignment

Extend the pipeline:

- Add **`order_date`** (synthetic column — e.g. `date_add` from `id` or random dates)
- **Partition** Parquet by date (`partitionBy("order_date")`)
- **Optimize:** broadcast a **small** lookup DataFrame joined to categories or users

**Bonus:** tune `repartition` / `coalesce` on write for file sizes.


## Interview Questions

1. How does Spark handle big data?
2. What is partitioning?
3. How do you optimize Spark pipelines?
4. Why use Parquet?


## What you just built

- **Volume-shaped** workload (synthetic)
- **Transforms**, **aggregates**, **windows**
- **Cache / repartition**, **Parquet** sink, **SQL** analytics

👉 Portfolio narrative: *“Spark pipeline at scale — tuning + columnar storage.”*


---

### Phase 4 complete

You can combine **distributed execution**, **SQL**, and **performance** trade-offs.

**Next — Phase 5:** **Apache Airflow** — schedules, DAGs, production workflow orchestration.

**Reality check:** batch Spark + orchestration is how many analytics platforms ship.


## Optional: stop Spark


In [ ]:
spark.stop()
print("Spark stopped.")


## Your tasks

- [ ] Run **Dataset setup** → **Analytics SQL** (expect ~1–2 minutes locally for `count` + writes).
- [ ] Inspect **`output_parquet/`** (part-*.parquet).
- [ ] **Practice:** top 10 users, AOV, counts per category.
- [ ] **Assignment:** date column + **partitioned** Parquet + broadcast join story.
- [ ] Draft **README bullet** for GitHub: inputs, guarantees, how you’d schedule next (Airflow teaser).
